In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from one.api import ONE

from load import pick_random_session_with_dataset, get_trial_binned_spikes_and_trials_df

one = ONE()  # uses your ONE config / credentials

eid = pick_random_session_with_dataset(
    one,
    dataset="spikes.times.npy",
    seed=1234,
)

trial_binned = get_trial_binned_spikes_and_trials_df(
    one=one,
    eid=eid,
    probe="probe00",
    align_event="stimOn_times",
    t_before=0.2,
    t_after=4,
    bin_size_s=0.1,
)

/Users/jmanley/Documents/code/decisivetimes/jason/9_neural_dsi/.venv/lib/python3.11/site-packages/one/util.py:436: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


,goCueTrigger_times,intervals_bpod_start,intervals_bpod_end,quiescencePeriod,stimOffTrigger_times,stimOff_times,stimOnTrigger_times,goCue_times,response_times,choice,...,feedbackType,rewardVolume,firstMovement_times,intervals_start,intervals_end,signed_contrast,stim_side,choice_str,correct,rt
0,10.010842,0.000000,3.571701,0.441705,12.564056,12.644020,9.945042,10.011820,10.564045,-1.0,...,-1.0,0.0,10.293719,9.492339,13.064060,-1.0000,-1,L,False,0.552225
1,14.074562,4.071899,6.494501,0.410087,15.486867,15.634084,13.974462,14.075442,14.486964,-1.0,...,1.0,1.5,14.307719,13.564261,15.986870,0.1250,1,L,True,0.411521
2,16.960872,6.934200,25.965202,0.446345,34.957623,35.043653,16.873072,16.961717,32.957618,-1.0,...,-1.0,0.0,32.718719,16.426571,35.457627,-0.2500,-1,L,False,15.995901
3,37.185531,26.477300,29.541002,0.467037,38.533436,38.610292,37.085431,37.186437,37.533532,-1.0,...,1.0,1.5,37.352719,35.969726,39.033440,1.0000,1,L,True,0.347096
4,40.054140,29.981799,81.947602,0.479629,90.940228,91.079482,39.954040,40.055046,88.940220,-1.0,...,-1.0,0.0,65.118719,39.474238,91.440232,0.0000,0,L,False,48.885174
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
703,3140.770748,3129.927898,3139.690401,0.473742,3148.693180,3148.756432,3140.670647,3140.771473,3146.693172,-1.0,...,-1.0,0.0,3146.460719,3139.430642,3149.193185,-0.1250,-1,L,False,5.921699
704,3151.145490,3141.009398,3144.787401,0.533004,3153.790202,3153.948573,3151.045390,3151.146323,3151.790193,1.0,...,-1.0,0.0,3151.153719,3150.512187,3154.290207,0.0000,0,R,False,0.643870
705,3156.306313,3146.050799,3163.132300,0.678769,3172.135170,3172.205925,3156.232513,3156.307234,3170.135163,-1.0,...,-1.0,0.0,3169.767719,3155.553611,3172.635173,0.0000,0,L,False,13.827929
706,3175.378580,3164.500099,3181.472700,0.631219,3190.475616,3190.539037,3175.278480,3175.379304,3188.475611,-1.0,...,-1.0,0.0,3188.166719,3174.002977,3190.975618,-0.0625,-1,L,False,13.096307


In [13]:
def premotor_population_activity(
    spike_times,
    spike_clusters,
    trials_df,
    stim_key="stimOn_times",
    move_key="firstMovement_times",
):
    """
    Compute pre-motor population activity vectors.

    Returns
    -------
    X : (n_trials, n_neurons) array
        Mean spike count per neuron between stimulus and movement.
    cluster_ids : (n_neurons,) array
        Cluster IDs corresponding to columns of X.
    """
    # Unique neurons (fixed column order)
    cluster_ids = np.unique(spike_clusters)
    n_neurons = cluster_ids.size

    # Map cluster id -> column index
    clu_to_col = {cid: i for i, cid in enumerate(cluster_ids)}

    X = np.zeros((len(trials_df), n_neurons), dtype=float)

    for i, row in trials_df.iterrows():
        t0 = row[stim_key]
        t1 = row[move_key]

        # Skip invalid trials
        if not np.isfinite(t0) or not np.isfinite(t1) or t1 <= t0:
            continue

        # Select spikes in pre-motor window
        mask = (spike_times >= t0) & (spike_times < t1)
        st = spike_times[mask]
        sc = spike_clusters[mask]

        if st.size == 0:
            continue

        duration = t1 - t0  # seconds

        # Count spikes per neuron
        for cid in np.unique(sc):
            X[i, clu_to_col[cid]] = np.sum(sc == cid) / duration

    return X, cluster_ids

In [14]:
X_premotor, cluster_ids = premotor_population_activity(
    spike_times,
    spike_clusters,
    trials_df,
)

NameError: name 'spike_times' is not defined

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from one.api import ONE

from load import (
    pick_random_session_with_dataset,
    load_spike_times_for_probe,
)

from brainbox.processing import bincount2D


# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------

one = ONE()

eid = pick_random_session_with_dataset(
    one,
    dataset="spikes.times.npy",
    seed=1234,
)

print("Using eid:", eid)

# ------------------------------------------------------------
# Load spikes (single probe for simplicity)
# ------------------------------------------------------------

spike_times, spike_clusters = load_spike_times_for_probe(
    one,
    eid=eid,
    probe="probe00",
)

print(f"Loaded {spike_times.size:,} spikes")

cluster_ids = np.unique(spike_clusters)
n_neurons = cluster_ids.size


# ------------------------------------------------------------
# Per-session binning
# ------------------------------------------------------------

bin_size = 0.02  # 20 ms

counts, bin_edges, clu_ids = bincount2D(
    spike_times,
    spike_clusters,
    xbin=bin_size,
)

# counts shape: (n_neurons, n_bins)
print("Session binned matrix:", counts.shape)


# ------------------------------------------------------------
# Load trials (simple version)
# ------------------------------------------------------------

trials = one.load_object(eid, "trials")

def trials_to_df_simple(trials):
    data = {}
    for k, v in trials.items():
        a = np.asarray(v)
        if a.ndim == 2 and a.shape[1] == 2:
            data[f"{k}_start"] = a[:, 0]
            data[f"{k}_end"] = a[:, 1]
        else:
            data[k] = a
    return pd.DataFrame(data)

trials_df = trials_to_df_simple(trials)

# Reaction time (optional sanity check)
trials_df["rt"] = (
    trials_df["firstMovement_times"] - trials_df["stimOn_times"]
)


# ------------------------------------------------------------
# Build pre-motor population vectors from session bins
# ------------------------------------------------------------

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

X_premotor = np.zeros((len(trials_df), n_neurons))

for i, row in trials_df.iterrows():
    t0 = row["stimOn_times"]
    t1 = row["firstMovement_times"]

    if not np.isfinite(t0) or not np.isfinite(t1) or t1 <= t0:
        continue

    # Find bin indices for this trial's window
    b0 = np.searchsorted(bin_centers, t0, side="left")
    b1 = np.searchsorted(bin_centers, t1, side="right")

    if b1 <= b0:
        continue

    # Mean firing rate per neuron
    duration = t1 - t0
    X_premotor[i] = counts[:, b0:b1].sum(axis=1) / duration


print("Pre-motor population matrix:", X_premotor.shape)

Using eid: cc45c568-c3b9-4f74-836e-c87762e898c8
Loaded 23,312,557 spikes
Session binned matrix: (677, 162423)


/Users/jmanley/Documents/code/decisivetimes/jason/9_neural_dsi/.venv/lib/python3.11/site-packages/one/util.py:436: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


Pre-motor population matrix: (715, 677)


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

In [27]:
# Labels: Left (-1) vs Right (+1)
choice = trials_df["choice"].to_numpy()

correct = (trials_df["feedbackType"] == 1).to_numpy()

# Keep only L/R trials
mask = np.isin(choice, [-1, 1])

X = X_premotor[mask]
y = (choice[mask] == 1).astype(int)  # 0 = Left, 1 = Right

correct = correct[mask]

print("Using trials:", X.shape[0])
print("Neurons:", X.shape[1])

Using trials: 715
Neurons: 677


In [38]:
X_train, X_test, y_train, y_test, correct_train, correct_test = train_test_split(
    X, y, correct, test_size=0.1, random_state=0, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),   # z-score neurons
    ("clf", LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
    )),
])

model.fit(X_train, y_train)

/Users/jmanley/Documents/code/decisivetimes/jason/9_neural_dsi/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with so

In [39]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Test accuracy: {acc:.3f}")
print(f"Test ROC AUC:  {auc:.3f}")

Test accuracy: 0.736
Test ROC AUC:  0.856


In [40]:
def eval_subset(name, mask):
    if mask.sum() < 10:
        print(f"{name}: too few trials")
        return

    acc = accuracy_score(y_test[mask], y_pred[mask])
    auc = roc_auc_score(y_test[mask], y_prob[mask])
    print(f"{name:>12s} | n={mask.sum():4d} | acc={acc:.3f} | auc={auc:.3f}")

mask_corr = correct_test == 1
mask_err  = correct_test == 0

eval_subset("Correct", mask_corr)
eval_subset("Incorrect", mask_err)

     Correct | n=  54 | acc=0.759 | auc=0.922
   Incorrect | n=  18 | acc=0.667 | auc=0.750
